# NEXORA Phase 4 — Predictive Analytics

Exploratory companion to the production pipelines in `models/`. **The production pipeline does not depend on this notebook** -- `python -m models.customer_churn`, `python -m models.project_risk`, `python -m models.payment_delay`, `python -m models.revenue_forecast`, and `python -m etl.validate_predictions` are all fully self-contained. This notebook shows the reasoning behind each model's design, most importantly the empirical investigation into *why* two of the four models find no real signal while the other two do -- see `docs/predictive_analytics.md` for the full write-up.

1. Customer risk feature exploration
2. Classification model comparison
3. Confusion matrix and ROC
4. Feature importance
5. Project risk feature exploration
6. Project model comparison
7. Payment delay analysis
8. Payment model evaluation
9. Revenue time-series exploration
10. Baseline comparison
11. Forecast visualization

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay, roc_curve

from mining.common import fetch_dataframe

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Customer risk feature exploration

The empirical investigation that shaped this model's design: check whether candidate behavioral features actually correlate with the churn outcome BEFORE trusting any model's apparent performance.

In [ ]:
churn_check = fetch_dataframe('''
    SELECT c.customer_id, c.product_usage_score, c.engagement_score, c.satisfaction_score,
           c.support_ticket_count_recent, c.avg_payment_delay_days, c.churn_risk_score, c360.total_revenue,
           COUNT(s.subscription_id) AS n_subs,
           MAX(IFF(s.status IN ('Cancelled','Expired'), 1, 0)) AS churned_any
    FROM CORE.DIM_CUSTOMER c
    JOIN ANALYTICS.VW_CUSTOMER_360 c360 ON c360.customer_id = c.customer_id
    LEFT JOIN CORE.FACT_SUBSCRIPTIONS s ON s.customer_key = c.customer_key
    WHERE c.is_current = TRUE AND c.customer_key <> -1
    GROUP BY c.customer_id, c.product_usage_score, c.engagement_score, c.satisfaction_score,
             c.support_ticket_count_recent, c.avg_payment_delay_days, c.churn_risk_score, c360.total_revenue
''')

# The count-artifact: P(any subscription cancelled) rises mechanically with subscription count
churn_check.groupby("N_SUBS")["CHURNED_ANY"].agg(["mean", "count"])

In [ ]:
# On the clean single-subscription population, every correlation collapses to ~0 -- confirms no real signal
single = churn_check[churn_check["N_SUBS"] == 1]
for col in ["PRODUCT_USAGE_SCORE", "ENGAGEMENT_SCORE", "SATISFACTION_SCORE", "AVG_PAYMENT_DELAY_DAYS", "TOTAL_REVENUE"]:
    print(f"{col}: corr={single[col].astype(float).corr(single['CHURNED_ANY'].astype(float)):.4f}")

## 2. Classification model comparison

In [ ]:
import json
with open("../artifacts/metrics/customer_churn.json") as f:
    churn_metrics = json.load(f)
pd.DataFrame(churn_metrics["algorithms_compared"]).T[["accuracy", "precision", "recall", "f1", "roc_auc"]]

## 3. Confusion matrix and ROC (customer churn model)

In [ ]:
selected = churn_metrics["selected_algorithm"]
cm = np.array(churn_metrics["algorithms_compared"][selected]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(cm, display_labels=["Not churned", "Churned"]).plot(ax=ax)
ax.set_title(f"Customer churn -- {selected} (test set)")
plt.show()
print("ROC-AUC ~0.5 confirms near-chance performance -- consistent with the correlation check in section 1.")

## 4. Feature importance (customer churn model)

In [ ]:
pd.Series(churn_metrics["feature_importance"]).sort_values(key=abs, ascending=False).plot(kind="barh", figsize=(7, 5))
plt.title(f"{selected} feature importance/coefficients -- customer churn"); plt.tight_layout(); plt.show()

## 5. Project risk feature exploration

Same diligence applied to the project model: check delay rate against segment, service, and budget before trusting any classifier's output.

In [ ]:
project_check = fetch_dataframe('''
    SELECT p.project_id, p.status, p.budget, c.segment
    FROM CORE.FACT_PROJECTS p JOIN CORE.DIM_CUSTOMER c ON c.customer_key = p.customer_key
    WHERE p.status IN ('Completed','Delayed')
''')
project_check["is_delayed"] = (project_check["STATUS"] == "Delayed").astype(int)
pd.crosstab(project_check["SEGMENT"], project_check["is_delayed"], normalize="index")

## 6. Project model comparison

In [ ]:
with open("../artifacts/metrics/project_risk.json") as f:
    project_metrics = json.load(f)
pd.DataFrame(project_metrics["algorithms_compared"]).T[["accuracy", "precision", "recall", "f1", "roc_auc"]]

## 7. Payment delay analysis

Unlike the two models above, payment delay shows genuine signal -- via a shared latent "health" factor in the generator (see `data_generator/customers.py`) that links usage/engagement/satisfaction to payment reliability.

In [ ]:
payment_check = fetch_dataframe('''
    SELECT c.customer_id, c.product_usage_score, AVG(p.days_late) AS avg_days_late
    FROM CORE.DIM_CUSTOMER c JOIN CORE.FACT_PAYMENTS p ON p.customer_key = c.customer_key
    WHERE c.is_current = TRUE AND c.customer_key <> -1
    GROUP BY c.customer_id, c.product_usage_score
''')
plt.figure(figsize=(6, 4))
plt.scatter(payment_check["PRODUCT_USAGE_SCORE"].astype(float), payment_check["AVG_DAYS_LATE"].astype(float), s=5, alpha=0.3)
plt.xlabel("product_usage_score"); plt.ylabel("avg days_late")
plt.title(f"corr = {payment_check['PRODUCT_USAGE_SCORE'].astype(float).corr(payment_check['AVG_DAYS_LATE'].astype(float)):.3f}")
plt.show()

## 8. Payment model evaluation

In [ ]:
with open("../artifacts/metrics/payment_delay.json") as f:
    payment_metrics = json.load(f)
display(pd.DataFrame(payment_metrics["algorithms_compared"]).T[["accuracy", "precision", "recall", "f1", "roc_auc"]])
pd.Series(payment_metrics["feature_importance"]).sort_values(ascending=False).head(10).plot(kind="barh", figsize=(7, 5))
plt.title("RandomForest feature importance -- payment delay"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## 9. Revenue time-series exploration

In [ ]:
daily = fetch_dataframe("SELECT full_date, invoice_amount FROM ANALYTICS.VW_REVENUE_TRENDS")
daily["FULL_DATE"] = pd.to_datetime(daily["FULL_DATE"])
daily["INVOICE_AMOUNT"] = daily["INVOICE_AMOUNT"].astype(float)
monthly = daily.groupby(daily["FULL_DATE"].dt.to_period("M"))["INVOICE_AMOUNT"].sum()
monthly.index = monthly.index.to_timestamp()

monthly.plot(figsize=(10, 4), marker="o", markersize=3)
plt.title("Monthly invoiced revenue (incl. partial trailing month)"); plt.ylabel("$"); plt.show()
print("~800x growth over 3 years -- a still-growing customer base, not seasonal demand. Last point is a partial month, excluded from modeling.")

## 10. Baseline comparison

In [ ]:
with open("../artifacts/metrics/revenue_forecast.json") as f:
    revenue_metrics = json.load(f)
pd.DataFrame(revenue_metrics["methods_compared"]).T.sort_values("rmse")

## 11. Forecast visualization

In [ ]:
forecast = fetch_dataframe("SELECT forecast_date, predicted_revenue, lower_bound, upper_bound FROM ANALYTICS.ML_REVENUE_FORECAST ORDER BY forecast_date")
forecast["FORECAST_DATE"] = pd.to_datetime(forecast["FORECAST_DATE"])
for c in ["PREDICTED_REVENUE", "LOWER_BOUND", "UPPER_BOUND"]:
    forecast[c] = forecast[c].astype(float)

complete_months = monthly.iloc[:-1] if monthly.index[-1].month == pd.Timestamp.now().month else monthly
plt.figure(figsize=(10, 5))
plt.plot(complete_months.index[-12:], complete_months.values[-12:], marker="o", label="Actual (last 12 complete months)")
plt.plot(forecast["FORECAST_DATE"], forecast["PREDICTED_REVENUE"], marker="o", color="orange", label=f"Forecast ({revenue_metrics['selected_method']})")
plt.fill_between(forecast["FORECAST_DATE"], forecast["LOWER_BOUND"], forecast["UPPER_BOUND"], alpha=0.2, color="orange", label="95% interval")
plt.legend(); plt.title("NEXORA monthly invoiced revenue: actual + 3-month forecast"); plt.ylabel("$"); plt.show()